<a href="https://colab.research.google.com/github/deeedaniel/gpt/blob/main/buildgpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

In [ ]:
text = "the cat sat on the mat. the cat ran. the dog sat on the mat too."
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size, chars)

14 [' ', '.', 'a', 'c', 'd', 'e', 'g', 'h', 'm', 'n', 'o', 'r', 's', 't']


In [ ]:
# Create mapping for characters to numbers
# Create encode and decode to convert text and numbers
stoi = {c: i for i,c in enumerate(chars)}
itos = {i: c for i,c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("cat"))
print(decode(encode("cat")))

[3, 2, 13]
cat


In [ ]:
# Convert our text into data (tensor)
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:20])

torch.Size([64]) torch.int64
tensor([13,  7,  5,  0,  3,  2, 13,  0, 12,  2, 13,  0, 10,  9,  0, 13,  7,  5,
         0,  8])


In [ ]:
# Split data into training data and validating data
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
print(len(train_data), len(val_data))

57 7


In [ ]:
block_size = 8
batch_size = 4

def get_batch(split):
  data = train_data if split == 'train' else val_data

  # pick x (4) random starting indices for our batches
  ix = torch.randint(len(data) - block_size, (batch_size,))

  # for all starting portions, get 8 consecutive characters
  # stack them up into 4 separate chunks for different batches
  # (4,8) 4 batch items, 8 chars each
  x = torch.stack([data[i:i+block_size] for i in ix])

  # y is the exact same but shifted forward by 1 to represent correct next char
  y = torch.stack([data[i+1:i+1+block_size] for i in ix])

  return x,y

In [ ]:
xb, yb = get_batch('train')
print(xb.shape, yb.shape)
print(xb[0])
print(yb[0])
print(decode(xb[0].tolist()))
print(decode(yb[0].tolist()))

torch.Size([4, 8]) torch.Size([4, 8])
tensor([13,  7,  5,  0,  3,  2, 13,  0])
tensor([ 7,  5,  0,  3,  2, 13,  0, 12])
the cat 
he cat s
